In [ ]:
!pip install pandas torch numpy sklearn statsmodels

In [2]:
import pandas as pd

df = pd.read_csv("apartments.csv")
print(df.head())
print(df.describe())
print(df.describe(include='object'))

TARGET = 'price'

         date      price  bedrooms  ...       city  statezip  price_per_sqft
0  2014-05-02   313000.0       3.0  ...  Shoreline  WA 98133          233.58
1  2014-05-02  2384000.0       5.0  ...    Seattle  WA 98119          653.15
2  2014-05-02   342000.0       3.0  ...       Kent  WA 98042          177.20
3  2014-05-02   420000.0       3.0  ...   Bellevue  WA 98008          210.00
4  2014-05-02   550000.0       4.0  ...    Redmond  WA 98052          283.51

[5 rows x 18 columns]
              price     bedrooms  ...  yr_renovated  price_per_sqft
count  4.600000e+03  4600.000000  ...   4600.000000     4600.000000
mean   5.519630e+05     3.400870  ...    808.608261      265.876209
std    5.638347e+05     0.908848  ...    979.414536      357.503362
min    0.000000e+00     0.000000  ...      0.000000        0.000000
25%    3.228750e+05     3.000000  ...      0.000000      180.817500
50%    4.609435e+05     3.000000  ...      0.000000      243.855000
75%    6.549625e+05     4.000000  ...  

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import os
import pandas as pd

def get_numeric_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=[np.number]).columns.tolist()

def get_categorical_columns(df: pd.DataFrame) -> list:
    return df.select_dtypes(include=["object", "category"]).columns.tolist()


def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """Koduje zmienne kategorialne za pomocą Label Encoding."""
    df_encoded = df.copy()
    categorical_cols = get_categorical_columns(df)
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df[col].astype(str))
    
    return df_encoded


def compute_correlation_matrix(df: pd.DataFrame, method: str = "pearson", include_categorical: bool = True) -> pd.DataFrame:
    """Oblicza macierz korelacji, opcjonalnie z uwzględnieniem zmiennych kategorialnych."""
    if include_categorical:
        df_encoded = encode_categorical(df)
        return df_encoded.corr(method=method)
    else:
        numeric_df = df.select_dtypes(include=[np.number])
        return numeric_df.corr(method=method)


def plot_correlation_matrix(
    corr_matrix: pd.DataFrame,
    output_path: str = "outputs/correlation_matrix.png",
    figsize: tuple = (18, 16),
    cmap: str = "coolwarm",
    title: str = "Macierz korelacji wszystkich zmiennych"
):
    plt.figure(figsize=figsize)
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 7}
    )
    
    plt.title(title, fontsize=16, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Macierz korelacji zapisana do: {output_path}")


def get_top_correlations(corr_matrix: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    corr_pairs = corr_matrix.unstack()
    
    corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) < corr_pairs.index.get_level_values(1)]
    
    corr_pairs = corr_pairs.reindex(corr_pairs.abs().sort_values(ascending=False).index)
    
    top_corr = pd.DataFrame({
        "Zmienna 1": [idx[0] for idx in corr_pairs.head(n).index],
        "Zmienna 2": [idx[1] for idx in corr_pairs.head(n).index],
        "Korelacja": corr_pairs.head(n).values
    })
    
    return top_corr

corr_matrix = compute_correlation_matrix(df, method="pearson", include_categorical=True)
plot_correlation_matrix(corr_matrix, output_path="outputs/correlation_matrix.png", title="Macierz korelacji wszystkich zmiennych")
print("Top 10 korelacji:")
print(get_top_correlations(corr_matrix, n=10))

df = df.drop(columns=['date', 'street'])
df.drop(columns=['price_per_sqft', 'sqft_living', 'sqft_above'])

Macierz korelacji zapisana do: outputs/correlation_matrix.png
Top 10 korelacji:
    Zmienna 1       Zmienna 2  Korelacja
0  sqft_above     sqft_living   0.876443
1       price  price_per_sqft   0.819279
2   bathrooms     sqft_living   0.761154
3   bathrooms      sqft_above   0.689918
4        city        statezip   0.683512
5    bedrooms     sqft_living   0.594884
6   bathrooms        bedrooms   0.545920
7      floors      sqft_above   0.522814
8   bathrooms          floors   0.486428
9    bedrooms      sqft_above   0.484705


KeyError: "['date', 'street'] not found in axis"

In [14]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

symbol = '+'

# Q("nazwa_zmiennej") - syntax umożliwiający posługiwanie się pełnymi nazwami kolumn do zdefiniowania modelu liniowego
# C(nazwa_zmiennej) - wskazanie, że dana zmienna jest zmienną kategorialną (jakościową)

categorical_vars = "".join([f'C(Q("{var}")) {symbol} ' for var in get_categorical_columns(df) if var != TARGET])
numeric_vars = "".join([f'{symbol if var != get_numeric_columns(df)[0] else ""} Q("{var}") ' for var in get_numeric_columns(df) if var != TARGET])

definition = f'Q("{TARGET}") ~ ' + categorical_vars + numeric_vars
print(definition)

stats_model = ols(definition, data=df).fit()
anova_result = sm.stats.anova_lm(stats_model, type=2)
print(anova_result)

"""
Wg testu ANOVA wszystkie zmienne są istotne
"""


Q("price") ~ C(Q("city")) + C(Q("statezip")) + + Q("bedrooms") + Q("bathrooms") + Q("sqft_living") + Q("sqft_lot") + Q("floors") + Q("waterfront") + Q("view") + Q("condition") + Q("sqft_above") + Q("sqft_basement") + Q("yr_built") + Q("yr_renovated") + Q("price_per_sqft") 
                         df        sum_sq  ...             F         PR(>F)
C(Q("city"))           43.0  1.571925e+14  ...    135.657873   0.000000e+00
C(Q("statezip"))       76.0  1.021328e+14  ...     49.869339   0.000000e+00
Q("bedrooms")           1.0  3.304641e+13  ...   1226.326194  9.067454e-238
Q("bathrooms")          1.0  5.333665e+13  ...   1979.281067   0.000000e+00
Q("sqft_living")        1.0  8.062124e+13  ...   2991.791134   0.000000e+00
Q("sqft_lot")           1.0  2.748396e+10  ...      1.019908   3.125952e-01
Q("floors")             1.0  9.579726e+11  ...     35.549610   2.676925e-09
Q("waterfront")         1.0  1.054986e+13  ...    391.497041   1.293781e-83
Q("view")               1.0  4.057750e+12 

'\nWg testu ANOVA wszystkie zmienne są istotne\n'

In [ ]:
def plot_scatter_plot(df, x, y):
    plt.scatter(df[x], df[y])
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(f"{x} & {y}")
    os.makedirs(f"outputs/scatterplots/", exist_ok=True)
    plt.savefig(f"outputs/scatterplots/{x} & {y}.png")
    plt.close()

for i in range(len(df.columns)):
    plot_scatter_plot(df, df.columns[i], TARGET)
    # for j in range(i + 1, len(df.columns)):
    #     plot_scatter_plot(df, df.columns[i], df.columns[j])


In [26]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

def create_preprocessor(df: pd.DataFrame, target_col: str):
    categorical = df.select_dtypes(include=["object"]).columns.tolist()

    if target_col in categorical:
        categorical.remove(target_col)

    for g in ["passed"]:
        if g in categorical:
            categorical.remove(g)

    numeric = df.select_dtypes(exclude=["object"]).columns.tolist()


    if target_col in numeric:
        numeric.remove(target_col)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
            ("num", StandardScaler(), numeric),
        ]
    )
    return preprocessor

RANDOM_STATE = 42
TEST_SIZE = int(df.__len__() * 0.3)

preprocessor = create_preprocessor(df, TARGET)

X = df.drop(columns=[TARGET])
y = df[TARGET]

train_x, test_x, train_y, test_y = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)

print(train_x.shape, test_x.shape)
train_x = preprocessor.fit_transform(X=train_x)
test_x = preprocessor.fit_transform(test_x)

import torch
print(train_x.shape, test_x.shape)
x_train_t = torch.tensor(train_x.toarray() if hasattr(train_x, "toarray") else train_x, dtype=torch.float32)
x_test_t = torch.tensor(test_x.toarray() if hasattr(test_x, "toarray") else test_x, dtype=torch.float32)
y_train_t = torch.tensor(train_y, dtype=torch.float32)
y_test_t = torch.tensor(test_y.to_numpy(), dtype=torch.float32)

print(x_train_t.shape, x_test_t.shape, y_train_t.shape, y_test_t.shape)

(3220, 15) (1380, 15)
(3220, 132) (1380, 128)
torch.Size([3220, 132]) torch.Size([1380, 128]) torch.Size([3220]) torch.Size([1380])


In [12]:

from sklearn.linear_model import LinearRegression, TheilSenRegressor
from sklearn.metrics import r2_score as R2

print(x_train_t.shape, x_test_t.shape)
lr_model = LinearRegression()
lr_model.fit(x_train_t, y_train_t)
y_pred = lr_model.predict(x_test_t)

r2 = R2(y_test_t, y_pred)
utheil = TheilSenRegressor().fit(x_test_t, y_test_t)


print("LR Model - R2: ", r2)
print(utheil.coef_)


torch.Size([3220, 132]) torch.Size([1380, 128])


ValueError: X has 128 features, but LinearRegression is expecting 132 features as input.

In [63]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import torch

def plot_roc_auc(y_test, y_score, output_path, model_name="Model"):
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'r--', label='Random Guess')

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Model')
    plt.legend()
    plt.savefig(output_path + " roc auc curve.png")
    plt.close()
    return roc_auc

def normalize_confusion_matrix(cm, norm='true'):
    """
    Normalize a confusion matrix.
    
    Parameters:
    cm (array-like): Confusion matrix to be normalized.
    norm (str): Type of normalization ('true', 'pred', 'all').
    
    Returns:
    ndarray: Normalized confusion matrix.
    """
    if norm == 'true':
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    elif norm == 'pred':
        cm_normalized = cm.astype('float') / cm.sum(axis=0)[np.newaxis, :]
    elif norm == 'all':
        cm_normalized = cm.astype('float') / cm.sum()
    else:
        raise ValueError("Unknown normalization type. Use 'true', 'pred', or 'all'.")
    
    return cm_normalized

def plot_confusion_matrix(y_test, y_pred, output_dir):
    cm = confusion_matrix(y_test, y_pred)
    normalized = normalize_confusion_matrix(cm, norm='true')
    plt.figure(figsize=(5, 4))
    sns.heatmap(normalized, annot=True, fmt=".2f", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_dir), exist_ok=True)
    plt.savefig(output_dir+" confusion matrix.png")
    plt.close()

def plot_training(total_loss, epochs, output_path):
    fig, ax = plt.subplots()
    
    ax.plot(epochs, total_loss, color='lightblue', linewidth=3)
    ax.set(xlabel="epochs", ylabel="loss")
    
    plt.savefig(output_path+" training.png")
    plt.close()


def train_model(model, x_train_t, y_train_t, epochs=200, lr=0.001, weight_decay=1e-5, logging_step=10, l1_param = 1e-7, output_path = f"outputs/training.png"):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    losses = []
    all_epochs = []
    best_loss = 999
    best_model_state = model.state_dict()
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(x_train_t)
        loss = criterion(outputs, y_train_t)
        total_loss = loss + sum(param.abs().sum() for param in model.parameters()) * l1_param
        if total_loss < best_loss:
            best_loss = total_loss
            best_model_state = model.state_dict()

        total_loss.backward()
        optimizer.step()
        if logging_step != -1 and epoch % logging_step == logging_step - 1:
            losses.append(total_loss.item())
            all_epochs.append(epoch+1)
            plot_training(epochs=all_epochs, total_loss=losses, output_path=output_path)
        model.load_state_dict(best_model_state)

def evaluate_model(model, x_test_t, y_test_t, output_dir, log = True):
    model.eval()
    criterion = nn.MSELoss()
    outputs = model(x_test_t)
    mse_loss = criterion(outputs, y_test_t)
    print(mse_loss.item())
    
            
    return acc




In [ ]:
from datetime import datetime

import torch.nn as nn

class DeepNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4, dropout=0.0):
        super(DeepNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_layer, 1)
        )
        

    def forward(self, x):
        x = self.net(x)
        return x.squeeze(dim=-1)
 

In [ ]:
deepnet = DeepNet(x_train_t.shape[1], hidden_layer=128, dropout=0.2)
output_path = f"outputs/{datetime.now().strftime("%H-%M-%S")}"
print(output_path)
train_model(deepnet, x_train_t, y_train_t, epochs=200, lr=0.001, weight_decay=1e-5, logging_step = 5, l1_param=0, output_path=output_path)
evaluate_model(deepnet, x_test_t, y_test_t, output_path)

In [65]:
import torch
import torch.nn as nn

class DeeperNet(nn.Module):
    def __init__(self, input: int = 26, hidden_layer: int = 26*4, count_of_layers=3):
        super(DeeperNet, self).__init__()
        dense_net = [[nn.Linear(hidden_layer, hidden_layer), nn.ReLU()] for i in range(count_of_layers - 1)]
        dense_net = [sublayer for layer in dense_net for sublayer in layer]
        self.net = nn.Sequential(
            nn.Linear(input, hidden_layer),
            nn.ReLU(),
            *dense_net,
            nn.Linear(hidden_layer, 1)
        )
        

    def forward(self, x):
        x = self.net(x)
        return x.squeeze(dim=-1)

output_path = f"outputs/{(datetime.now().strftime("%H-%M-%S"))}"
print(output_path)
deepernet = DeeperNet(x_train_t.shape[1], hidden_layer=256, count_of_layers=10)
train_model(deepernet, x_train_t, y_train_t, epochs=50, lr=0.001, weight_decay=1e-5, logging_step = 5, output_path=output_path, l1_param=0)
evaluate_model(deepernet, x_test_t, y_test_t, output_path)

outputs/12-28-54


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1380x128 and 132x256)

In [ ]:
import optuna

def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 10, 1000)
    dropout = trial.suggest_float("dropout", 0, 1)
    model = DeepNet(x_train_t.shape[1], hidden_layer=hidden_size, dropout=dropout)
    train_model(model, x_train_t, y_train_t, epochs=100, l1_param=0, logging_step=-1)
    return evaluate_model(model, x_test_t, y_test_t, output_path, log = False)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)
print(study.best_params)